# 00 Prepare Datasets

This notebook creates the paper-facing final dataset layer used by all later notebooks.

**Inputs.** It reads the processed Alpha and Beta source datasets configured in `config/experiment_config.yaml`.

**Outputs.** It writes `dataset/final/dataset_alpha.parquet`, `dataset/final/dataset_beta.parquet`, `dataset/final/dataset_gamma.parquet`, final-dataset summaries, Gamma selection rankings, and checksums.

**Key decisions.** Alpha keeps the full train/test period, Beta is filtered to 2023-10-01 through 2024-09-30, and Gamma is selected as one Beta site for the forecast-impact case study. Site IDs are renamed only in this final layer, so paper-facing notebooks use `alpha_*` and `beta_*` IDs while processed source files remain unchanged.

**When to rerun.** Rerun this notebook when the processed source datasets change, when the manual oracle-reviewed Beta file replaces the provisional labels, or when you want to select a different Gamma site.


## 1. Imports And Paths

Resolve the article root, load the `journal_v2` config, and show output locations before writing anything. The helper functions use these paths consistently, so later notebooks can assume the same directory structure.


In [ ]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

resolved_paths = {
    "alpha_final": article_root / cfg["paths"]["alpha_dataset_path"],
    "beta_final": article_root / cfg["paths"]["beta_dataset_path"],
    "gamma_final": article_root / cfg["paths"]["gamma_dataset_path"],
    "dataset_final_dir": paths.final,
}
resolved_paths


## 2. Optional Gamma Site Override

This cell controls the Gamma case-study site. Leave `GAMMA_SITE_OVERRIDE = None` for automatic selection using the configured ranking, or set a specific `beta_*` site after inspecting the Gamma candidate table. The override only affects the final Gamma dataset, not Alpha or Beta.


In [ ]:
# Optional override after reviewing Gamma candidates.
# Set to None to use config auto-selection.
GAMMA_SITE_OVERRIDE = None  # e.g. "beta_B"


## 3. Build Final Datasets

`h.run_prepare_datasets()` loads the processed sources, validates the seven-column schema, renames final-layer site IDs, filters Beta to the one-year window, selects Gamma, writes the final Parquet files, and records summaries/checksums. If this helper fails, inspect the validation table in the next section before moving on to later notebooks.


In [ ]:
# This is the only notebook that writes the final Alpha/Beta/Gamma dataset layer.
result = h.run_prepare_datasets(article_root, gamma_site_override=GAMMA_SITE_OVERRIDE)
final_summary = result["final_summary"]
final_summary


## 4. Review Site Rankings

These rankings make the automatic choices auditable. Alpha rankings define the top leave-one-station-out folds for Notebook 2. Beta rankings explain the Gamma site selection by showing the sites with the largest apparent RPF impact under the current labels.


In [ ]:
# Recompute rankings from the in-memory result so the displayed choices match the files just written.
alpha_rank = h.site_rpf_summary(result["alpha"], "Alpha").sort_values(
    ["rpf_days", "rpf_intervals", "substation_id"],
    ascending=[False, False, True],
)
gamma_rank = result["gamma_rankings"]
print(f"Selected Gamma site: {result['gamma_site']}")
display(alpha_rank.head(10))
display(gamma_rank.head(10))


## 5. Validation Summary

The validation summary confirms that the final datasets match the v2 assumptions: Alpha covers the train/test windows, Beta contains exactly the one-year review window, Gamma contains exactly one site, and all final payloads preserve the original seven-column schema.


In [ ]:
validation = result["validation"]
validation
